In [1]:
!pip install -q transformers datasets trl accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 11.2 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from trl import SFTTrainer, SFTConfig

# 1. KHỞI TẠO TOKENIZER VÀ THÊM TỪ VỰNG MỚI
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

special_tokens_dict = {'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<tools>', '</tools>', '<tool_call>', '</tool_call>']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
print(f"Đã thêm {num_added_toks} tokens đặc biệt vào Tokenizer.")

# 2. KHỞI TẠO MÔ HÌNH VÀ RESIZE EMBEDDINGS
model = GPT2LMHeadModel.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

# 3. TẢI DỮ LIỆU (TRAIN VÀ VAL)
# LƯU Ý: Thay đổi 2 đường dẫn dưới đây trỏ tới đúng file train và val của bạn trên Kaggle
data_files = {
    "train": "/kaggle/input/datasets/hykhangg/jsonlagent-copyfix/train_tool_data_copyfix.jsonl",
    "validation": "/kaggle/input/datasets/hykhangg/jsonlagent-copyfix/valid_tool_data_copyfix.jsonl"
}
dataset = load_dataset("json", data_files=data_files)

train_dataset = dataset["train"]
val_dataset = dataset["validation"]
print(f"Số dòng Train: {len(train_dataset)} | Số dòng Val: {len(val_dataset)}")

# 4. CẤU HÌNH HUẤN LUYỆN ĐỈNH CAO
training_args = SFTConfig(
    output_dir="./gpt2-small-agent",
    per_device_train_batch_size=2,   
    gradient_accumulation_steps=4,   
    learning_rate=3e-5,              
    num_train_epochs=5,                # Tăng lên 5 vòng để học kỹ tham số
    logging_steps=50,
    
    # --- CẤU HÌNH ĐÁNH GIÁ (VALIDATION) ---
    eval_strategy="steps",             # Đánh giá theo số bước
    eval_steps=500,                    # Cứ 500 bước thì làm bài test trên tập Val 1 lần
    save_steps=500,                    # Lưu checkpoint cùng lúc với lúc đánh giá
    load_best_model_at_end=True,       # Tự động lấy checkpoint có điểm Val tốt nhất ở cuối
    metric_for_best_model="eval_loss", # Tiêu chí chọn là độ sai lệch (loss) trên tập Val thấp nhất
    save_total_limit=2,                # Chỉ giữ 2 checkpoint trên ổ cứng để chống tràn RAM Kaggle
    
    fp16=True,                       
    optim="adamw_torch",
    report_to="none",
    
    dataset_text_field="text",       
    max_length=512,                  
    packing=False                    
)

# 5. CHẠY SFT TRAINER
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,          # Nạp tập Val vào "trường thi"
    args=training_args,
)

print("Bắt đầu quá trình huấn luyện có kiểm duyệt (Validation)...")
trainer.train()

# 6. LƯU MÔ HÌNH XUẤT SẮC NHẤT
model.save_pretrained("./final_gpt2_small_agent")
tokenizer.save_pretrained("./final_gpt2_small_agent")
print("Đã lưu thành công phiên bản GPT-2 xịn nhất!")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Đã thêm 6 tokens đặc biệt vào Tokenizer.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu thành công phiên bản GPT-2 xịn nhất!
